### 1. Install Dependencies
We need `yfinance` for data retrieval and `mplfinance` for generating professional candlestick charts without axes or grids.

In [1]:
!pip install yfinance mplfinance tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.0 MB/s eta 0:00:00


### 2. Mount Google Drive
This cell mounts your Drive so we can store the final dataset persistently.

In [2]:
from google.colab import drive
import os
#
drive.mount('/content/drive')

# Define paths
BASE_DRIVE_PATH = '/content/drive/MyDrive/stock_dataset/raw_images/'
TEMP_LOCAL_PATH = '/content/temp_images/'

# Create local temp directory
os.makedirs(TEMP_LOCAL_PATH, exist_ok=True)

Mounted at /content/drive


### 3. Core Pipeline Implementation
This cell contains the modular functions for data fetching, preprocessing, windowing, and image generation.

In [6]:
import yfinance as yf
import pandas as pd
import numpy as np
import mplfinance as mpf
from PIL import Image
from tqdm import tqdm
import shutil
import io
import os
import matplotlib.pyplot as plt

# STOCKS = [
#     # ─── LARGE CAP ───────────────────────────────────────────────
#     'RELIANCE.NS', 'HDFCBANK.NS', 'TCS.NS', 'ICICIBANK.NS', 'INFY.NS',
#     'HINDUNILVR.NS', 'ITC.NS', 'SBIN.NS', 'BHARTIARTL.NS', 'KOTAKBANK.NS',
#     'AXISBANK.NS', 'LT.NS', 'MARUTI.NS', 'SUNPHARMA.NS', 'ULTRACEMCO.NS',

#     # ─── MID CAP ─────────────────────────────────────────────────
#     'LTTS.NS', 'VOLTAS.NS', 'TATAELXSI.NS', 'ASHOKLEY.NS', 'MUTHOOTFIN.NS',
#     'COFORGE.NS', 'AARTIIND.NS', 'DEEPAKNTR.NS', 'PIIND.NS', 'SRF.NS',
#     'MPHASIS.NS', 'PERSISTENT.NS', 'LTIM.NS', 'POLYCAB.NS', 'ASTRAL.NS',
#     'PAGEIND.NS', 'BHARATFORG.NS', 'CUMMINSIND.NS', 'APLAPOLLO.NS', 'TORNTPHARM.NS',
#     'ALKEM.NS', 'BIOCON.NS', 'LAURUSLABS.NS', 'FEDERALBNK.NS', 'AUBANK.NS',

#     # ─── SMALL CAP ───────────────────────────────────────────────
#     'IRCTC.NS', 'CDSL.NS', 'NAZARA.NS', 'ROUTE.NS', 'TANLA.NS',
#     'AFFLE.NS', 'DATAPATTNS.NS', 'KNRCON.NS', 'RVNL.NS', 'IRCON.NS',
# ]


#take 2
# STOCKS = [
#     # ─── MID CAP (remaining 20) ──────────────────────────────────
#     'COFORGE.NS', 'AARTIIND.NS', 'DEEPAKNTR.NS', 'PIIND.NS', 'SRF.NS',
#     'MPHASIS.NS', 'PERSISTENT.NS', 'LTIM.NS', 'POLYCAB.NS', 'ASTRAL.NS',
#     'PAGEIND.NS', 'BHARATFORG.NS', 'CUMMINSIND.NS', 'APLAPOLLO.NS', 'TORNTPHARM.NS',
#     'ALKEM.NS', 'BIOCON.NS', 'LAURUSLABS.NS', 'FEDERALBNK.NS', 'AUBANK.NS',

#     # ─── SMALL CAP (all 10) ──────────────────────────────────────
#     'IRCTC.NS', 'CDSL.NS', 'NAZARA.NS', 'ROUTE.NS', 'TANLA.NS',
#     'AFFLE.NS', 'DATAPATTNS.NS', 'KNRCON.NS', 'RVNL.NS', 'IRCON.NS',
# ]

# #take 3
# STOCKS = [
#     # ─── SMALL CAP (all to fetch) ────────────────────────────────
#     'CDSL.NS', 'NAZARA.NS', 'ROUTE.NS', 'TANLA.NS',
#     'AFFLE.NS', 'DATAPATTNS.NS', 'KNRCON.NS', 'RVNL.NS', 'IRCON.NS',
# ]

#take 4
STOCKS = [
    # ─── LARGE CAP (missing) ──────────────────────────────────
    'POLYCAB.NS',        # Should be mid-cap, but not in image
    'PAGEIND.NS',        # Not in image
    'BHARATFORG.NS',     # Not in image
    'APLAPOLLO.NS',      # Not in image
    'TORNTPHARM.NS',     # Not in image
    'ALKEM.NS',          # Not in image
    'LAURUSLABS.NS',     # Not in image

    # ─── SMALL CAP (all present in image) ──────────────────────
    'KNRCON.NS',         # Not in image
]

def fetch_stock_data(ticker, interval='1h', period='60d'):
    try:
        df = yf.download(ticker, period=period, interval=interval, progress=False, auto_adjust=True)
        if df.empty: return None
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        return df
    except Exception as e:
        print(f"Error fetching {ticker}: {e}")
        return None

# def fetch_stock_data(ticker):
#     all_data = []

#     periods = ["60d"]

#     for p in periods:
#         try:
#             df = yf.download(
#                 ticker,
#                 period=p,
#                 interval="1h",
#                 progress=False,
#                 auto_adjust=True
#             )
#             if not df.empty:
#                 if isinstance(df.columns, pd.MultiIndex):
#                     df.columns = df.columns.get_level_values(0)
#                 all_data.append(df)
#         except:
#             continue

#     if len(all_data) == 0:
#         return None

#     df = pd.concat(all_data)
#     df = df[~df.index.duplicated(keep='first')]

#     return df

def preprocess_data(df):
    df = df.tz_convert('Asia/Kolkata')
    cols = ['Open', 'High', 'Low', 'Close']
    df = df[cols].copy()
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df.dropna(inplace=True)
    df = df.between_time('09:15', '15:30')
    df.sort_index(inplace=True)
    return df


def create_normalized_windows(df, window_size=20):
    windows = []
    for i in range(len(df) - window_size + 1):
        window = df.iloc[i : i + window_size].copy()
        first_close = float(window['Close'].iloc[0])
        if first_close == 0: continue
        for col in ['Open', 'High', 'Low', 'Close']:
            window[col] = window[col] / first_close
        windows.append(window)
    return windows

def save_window_as_image(window, ticker, idx, output_dir):
    """
    Optimized image generation for ViT:
    Removes margins, increases candle thickness, and ensures full canvas usage.
    """
    # Create a figure with specific size (2.24 inches at 100 DPI = 224 pixels)
    fig, ax = plt.subplots(figsize=(2.24, 2.24), dpi=100)
    # ax = fig.add_subplot(1,1,1)
    ax.set_position([0, 0, 1, 1])
    ax.margins(x=0, y=0)

    # Custom style to increase visibility for Deep Learning
    # width_config: increases body width; line_width: increases wick thickness
    market_colors = mpf.make_marketcolors(
        up='green',
        down='red',
        edge='inherit',
        wick='black',
        volume='inherit'
        )
    custom_style = mpf.make_mpf_style(marketcolors=market_colors, gridstyle='', y_on_right=False, facecolor='white')

    try:
        mpf.plot(window,
                 type='candle',
                 ax=ax,
                 style=custom_style,
                 axisoff=True,
                 update_width_config=dict(candle_linewidth=2.0, candle_width=0.9),
                 scale_width_adjustment=dict(candle=1.2),
                 xlim=(0, len(window)))

        # Remove all padding/margins to fill the entire image area
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

        stock_name = ticker.replace('.NS', '')
        stock_folder = os.path.join(output_dir, stock_name)
        os.makedirs(stock_folder, exist_ok=True)
        img_path = os.path.join(stock_folder, f"{stock_name}_{idx}.png")

        # Save directly to disk (efficient for 224x224 targeting)
        fig.savefig(img_path, dpi=200, format='png', pad_inches=0)

    except Exception as e:
        print(f"Error generating image {idx} for {ticker}: {e}")
    finally:
        plt.close(fig)

### 4. Execute Processing Loop
We iterate through stocks, process windows, and finally sync the local images to Google Drive.

In [7]:
for ticker in STOCKS:
    print(f"Processing {ticker}...")
    raw_data = fetch_stock_data(ticker)
    if raw_data is None: continue

    clean_data = preprocess_data(raw_data)
    windows = create_normalized_windows(clean_data)

    for i, win in enumerate(tqdm(windows, desc=f"Generating Images for {ticker}")):
        save_window_as_image(win, ticker, i, TEMP_LOCAL_PATH)

print("\nAll images generated locally. Moving to Google Drive...")

if not os.path.exists(BASE_DRIVE_PATH):
    os.makedirs(BASE_DRIVE_PATH)

for folder in os.listdir(TEMP_LOCAL_PATH):
    src = os.path.join(TEMP_LOCAL_PATH, folder)
    dst = os.path.join(BASE_DRIVE_PATH, folder)
    if os.path.isdir(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.move(src, dst)

print(f"Dataset successfully stored at: {BASE_DRIVE_PATH}")

Processing POLYCAB.NS...


Generating Images for POLYCAB.NS: 100%|██████████| 387/387 [00:25<00:00, 15.43it/s]


Processing PAGEIND.NS...


Generating Images for PAGEIND.NS:  45%|████▌     | 175/388 [00:10<00:13, 15.96it/s]


KeyboardInterrupt: 